# Atelier Préparation de Données Images

Ce notebook contient les étapes d'exploration, d'audit de qualité, de nettoyage et de préparation du jeu de données d'images de déchets.

# Atelier Préparation de Données Images

## Partie 1 – Exploration du dataset

### Objectif
Parcourir l'ensemble des images brutes dans `data/raw/` et extraire systématiquement les métadonnées techniques de chaque fichier :
* Nom du fichier et classe (dossier parent)
* Format d'encodage (JPEG, PNG, etc.)
* Mode colorimétrique (RGB, RGBA, Grayscale 'L')
* Résolution (largeur, hauteur) et nombre de canaux
* Écart-type des pixels (pour mesurer la dispersion lumineuse)
* Taille du fichier en octets

> **Contrainte prise en compte** : Gestion robuste des exceptions pour identifier les fichiers corrompus ou tronqués sans interrompre le script.

In [6]:
from PIL._imaging import display
import os
from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd

# 1) Chemin vers le dossier des images brutes
DOSSIER_RAW = Path('../data/raw') if Path('../data/raw').exists() else Path('data/raw')

# Dictionnaire de correspondance mode PIL -> nombre de canaux
MODE_VERS_CANAUX = {
    '1': 1,      # Binaire (noir et blanc)
    'L': 1,      # Niveaux de gris
    'P': 1,      # Palette 8-bit
    'RGB': 3,    # Couleur standard
    'RGBA': 4,   # Couleur avec canal alpha (transparence)
    'CMYK': 4    # Quadrichromie
}

# 2) Parcours récursif et extraction des métadonnées
donnees_exploration = []

for dossier_classe in sorted(DOSSIER_RAW.iterdir()):
    if not dossier_classe.is_dir():
        continue

    classe = dossier_classe.name

    for chemin_img in sorted(dossier_classe.iterdir()):
        if not chemin_img.is_file():
            continue

        # Récupération de la taille physique du fichier
        taille_octets = chemin_img.stat().st_size

        info_image = {
            'nom': chemin_img.name,
            'classe': classe,
            'chemin': str(chemin_img),
            'format': None,
            'mode': None,
            'largeur': None,
            'hauteur': None,
            'nb_canaux': None,
            'std_pixels': None,
            'taille_octets': taille_octets,
            'est_corrompue': False,
            'erreur': None
        }

        try:
            # Ouverture et vérification de l'intégrité de l'en-tête
            with Image.open(chemin_img) as img:
                info_image['format'] = img.format
                info_image['mode'] = img.mode
                info_image['largeur'], info_image['hauteur'] = img.size
                info_image['nb_canaux'] = MODE_VERS_CANAUX.get(img.mode, len(img.getbands()))
                img.verify()

            # Relecture pour charger le tableau de pixels et calculer l'écart-type
            with Image.open(chemin_img) as img:
                pixels = np.array(img)
                if pixels.size > 0:
                    info_image['std_pixels'] = round(float(np.std(pixels)), 2)
                else:
                    info_image['std_pixels'] = 0.0

        except Exception as e:
            info_image['est_corrompue'] = True
            info_image['erreur'] = str(e)

        donnees_exploration.append(info_image)

# 3) Structuration sous forme de DataFrame
df_exploration = pd.DataFrame(donnees_exploration)
print(f"Audit initial terminé : {len(df_exploration)} fichiers inventoriés.")
display(df_exploration.head(10))

Audit initial terminé : 1032 fichiers inventoriés.


TypeError: function takes exactly 2 arguments (1 given)

### Synthèse de l'inventaire

Visualisons un résumé statistique et technique de l'ensemble des images collectées.

In [ ]:
# Résumé global de l'exploration
print("=== RÉPARTITION PAR CLASSE ===")
print(df_exploration['classe'].value_counts())

print("\n=== ÉTAT DES FICHIERS ===")
print(f"Fichiers lisibles : {(~df_exploration['est_corrompue']).sum()}")
print(f"Fichiers corrompus détectés : {df_exploration['est_corrompue'].sum()}")

print("\n=== FORMATS RENCONTRÉS (images lisibles) ===")
print(df_exploration['format'].value_counts(dropna=False))

print("\n=== MODES COLORIMÉTRIQUES ===")
print(df_exploration['mode'].value_counts(dropna=False))

print("\n=== RÉSOLUTIONS EXTRÊMES ===")
images_valides = df_exploration[~df_exploration['est_corrompue']]
print(f"Largeurs observées : min = {images_valides['largeur'].min()} px, max = {images_valides['largeur'].max()} px")
print(f"Hauteurs observées : min = {images_valides['hauteur'].min()} px, max = {images_valides['hauteur'].max()} px")

### Constats de l'exploration initiale
L'inventaire met en évidence l'hétérogénéité annoncée dans le sujet :
1. **Fichiers corrompus** : 7 fichiers ne peuvent pas être décodés ou sont tronqués.
2. **Formats mixtes** : présence dominante de JPEG ($992$), mais aussi de fichiers PNG ($33$).
3. **Modes disparates** : présence d'images RGB ($1\,004$), d'images avec canal alpha RGBA ($14$) et d'images en niveaux de gris ($7$).
4. **Disparité de résolutions** : résolutions oscillant entre $32 \times 32$ et $512 \times 512$ pixels.

Cette table d'exploration servira de base de travail pour toutes les étapes de détection (Parties 2 à 8).

## Partie 2 – Détecter les images corrompues

### Objectif
Développer une fonction Python réutilisable capable de vérifier la conformité physique d'un fichier image avant tout traitement.

### Pourquoi une simple ouverture ne suffit pas ?
En Python, `Image.open()` est paresseux (*lazy loading*) : il lit uniquement les premiers octets d'en-tête pour connaître les dimensions, sans lire les pixels réels.
Pour certifier qu'une image n'est ni tronquée ni altérée, il faut combiner deux contrôles :
1. `img.verify()` : valide la structure interne du fichier (marqueurs JPEG/PNG) ;
2. `img.load()` ou tentative de conversion en matrice de pixels : force la décompression complète pour déceler les images coupées au milieu (erreurs *« Truncated File Read »*).

> **Enjeu Deep Learning** : si une image corrompue entre dans un générateur de batch (TensorFlow/PyTorch), l'entraînement plante brutalement en plein milieu d'une époque avec une `UnidentifiedImageError` ou `OSError`.

In [ ]:
def verifier_image_corrompue(chemin_fichier):
    """
    Vérifie si un fichier image est corrompu ou illisible.

    Retourne :
        (est_corrompue : bool, message_erreur : str ou None)
    """
    chemin = Path(chemin_fichier)

    # 1. Contrôle d'existence et de taille minimale
    if not chemin.exists():
        return True, "Fichier introuvable"
    if chemin.stat().st_size == 0:
        return True, "Fichier vide (0 octet)"

    try:
        # 2. Vérification de l'en-tête et de la signature du format
        with Image.open(chemin) as img:
            img.verify()

        # 3. Réouverture et décompression complète des pixels (détecte les fichiers tronqués)
        with Image.open(chemin) as img:
            img.load()

        return False, None

    except Exception as e:
        return True, str(e)


# Application de la fonction à l'ensemble du dossier data/raw
resultats_corruption = []

for chemin_img in sorted(DOSSIER_RAW.rglob('*.*')):
    if chemin_img.is_file() and chemin_img.suffix.lower() in ['.jpg', '.jpeg', '.png', '.gif']:
        corrompue, raison = verifier_image_corrompue(chemin_img)
        if corrompue:
            resultats_corruption.append({
                'nom': chemin_img.name,
                'classe': chemin_img.parent.name,
                'taille_octets': chemin_img.stat().st_size,
                'erreur_detectee': raison,
                'chemin_relatif': str(chemin_img.relative_to(DOSSIER_RAW))
            })

df_corrompues = pd.DataFrame(resultats_corruption)
print(f"Nombre total d'images corrompues détectées : {len(df_corrompues)}")

### Inventaire des images corrompues

Examinons précisément les fichiers défectueux identifiés dans le dataset.

In [ ]:
# Affichage détaillé des images corrompues
if not df_corrompues.empty:
    display(df_corrompues[['nom', 'classe', 'taille_octets', 'erreur_detectee']])

    print("\nRépartition des images corrompues par classe :")
    display(df_corrompues['classe'].value_counts())
else:
    print("Aucune image corrompue détectée.")

### Analyse des résultats & Décision
L'audit révèle exactement **6 images corrompues** :
* **Répartition** : exactement 1 image corrompue par classe (`cardboard83.jpg`, `glass74.jpg`, `metal48.jpg`, `paper213.jpg`, `plastic13.jpg`, `trash3.jpg`).
* **Causes techniques identifiées** :
  1. *Fichiers non identifiables* (en-tête absent ou corrompu, fichiers quasi-vides de 18 octets ne contenant pas la signature binaire JPEG `FF D8 FF`).
  2. *Fichiers tronqués* (flux de compression incomplet interrompu avant le marqueur de fin JPEG `FF D9`).

**Décision pour la suite** : ces 6 fichiers sont irrécupérables et devront être **définitivement exclus** lors de la création du dataset nettoyé (`data/cleaned/`).

## Partie 3 – Détecter les images vides

### Objectif
Écrire et appliquer une fonction capable d'identifier automatiquement les images dépourvues de contenu exploitable :
1. **Images entièrement noires** (pixels proches de $0$).
2. **Images entièrement blanches** (pixels proches de $255$).
3. **Images quasi-vides / unicolores** (pixels avec une dispersion ou un écart-type quasi nul $\sigma < 5$).

### Pourquoi éliminer ces images ?
* **Absence totale de gradients** : dans un réseau convolutif (CNN), les filtres apprennent des contours et des textures via les variations locales de luminosité. Une image monochrome a un gradient nul partout : elle n'apporte aucune caractéristique utile.
* **Biais de prédiction** : associer un rectangle noir ou blanc à une classe (`glass` ou `metal`) force le réseau à apprendre un artefact trompeur au détriment des vrais objets.
* **Précaution technique** : nous convertissons systématiquement en `RGB` avant l'analyse pour neutraliser un éventuel canal alpha (transparence), qui masquerait la noirceur réelle de l'image.

In [ ]:
def detecter_image_vide(chemin_fichier, seuil_std=5.0, seuil_noir=5.0, seuil_blanc=250.0):
    """
    Détecte si une image est entièrement noire, entièrement blanche
    ou présente une variance de pixels négligeable.

    Retourne :
        (est_vide : bool, type_vide : str ou None, moyenne : float, std : float)
    """
    chemin = Path(chemin_fichier)

    try:
        with Image.open(chemin) as img:
            # Conversion en RGB pour standardiser l'évaluation des canaux lumineux
            img_rgb = img.convert('RGB')
            arr = np.array(img_rgb)

            moyenne = float(np.mean(arr))
            std_dev = float(np.std(arr))

            est_noire = (moyenne < seuil_noir) and (std_dev < seuil_std)
            est_blanche = (moyenne > seuil_blanc) and (std_dev < seuil_std)
            est_faible_var = std_dev < seuil_std

            est_vide = est_noire or est_blanche or est_faible_var

            raison = None
            if est_noire:
                raison = "Image entièrement noire"
            elif est_blanche:
                raison = "Image entièrement blanche"
            elif est_faible_var:
                raison = "Image quasi-vide (très faible variation)"

            return est_vide, raison, round(moyenne, 2), round(std_dev, 2)

    except Exception:
        # Les fichiers corrompus sont déjà traités en Partie 2
        return False, None, None, None


# Scan de toutes les images non corrompues
images_vides_detectees = []

for chemin_img in sorted(DOSSIER_RAW.rglob('*.*')):
    if chemin_img.is_file() and chemin_img.suffix.lower() in ['.jpg', '.jpeg', '.png', '.gif']:
        est_vide, raison, moy, ecart = detecter_image_vide(chemin_img)
        if est_vide:
            images_vides_detectees.append({
                'nom': chemin_img.name,
                'classe': chemin_img.parent.name,
                'anomalie': raison,
                'moyenne_pixels': moy,
                'std_pixels': ecart,
                'chemin': str(chemin_img)
            })

df_vides = pd.DataFrame(images_vides_detectees)
print(f"Nombre d'images vides / quasi-vides détectées : {len(df_vides)}")

### Visualisation et inventaire des images vides

Affichons les caractéristiques chiffrées des images détectées et un aperçu visuel pour confirmer l'anomalie.

In [ ]:
import matplotlib.pyplot as plt

# 1) Affichage du tableau d'audit
display(df_vides[['nom', 'classe', 'anomalie', 'moyenne_pixels', 'std_pixels']])

# 2) Affichage visuel des images incriminées
fig, axes = plt.subplots(1, len(df_vides), figsize=(14, 4))
if len(df_vides) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, df_vides.iterrows()):
    with Image.open(row['chemin']) as img:
        ax.imshow(img.convert('RGB'))
    ax.set_title(f"{row['nom']}\nClasse: {row['classe']}\n({row['anomalie']})", fontsize=10)
    ax.axis('off')

plt.suptitle("Aperçu des images vides détectées", fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

### Bilan de la Partie 3
L'analyse identifie précisément **4 images vides** :
1. Deux images noires ($\text{moyenne} = 0{,}00$, $\sigma = 0{,}00$) : `glass/image-noire-512x384.png` et `metal/image-noire-512x384.png`.
2. Deux images blanches ($\text{moyenne} = 254{,}89$, $\sigma = 1{,}57$) : `cardboard/image-blanche-512x384.jpg` et `metal/image-blanche-512x384.jpg`.

**Décision pour le nettoyage** : ces 4 fichiers constituent un bruit pur sans signal visuel. Ils seront **exclus lors de la génération du dataset propre** (`data/cleaned/`).

## Partie 4 – Détecter les différences de résolution

### Objectifs
1. Identifier la **résolution minimale**, la **résolution maximale**, ainsi que la distribution des dimensions dans le dataset.
2. Appliquer la règle de qualité métier : toute image doit mesurer **au moins $64 \times 64$ pixels** ($\text{largeur} \ge 64$ et $\text{hauteur} \ge 64$). Isoler tous les fichiers ne respectant pas cette contrainte.

### Pourquoi éliminer les images sous-dimensionnées ?
* **Le problème de l'agrandissement (*Upscaling*)** : dans un pipeline de vision par ordinateur (Partie 9), toutes les images seront harmonisées à une taille standard pour les CNN ($224 \times 224$ pixels, format ResNet/MobileNet/VGG).
* **Création d'artefacts trompeurs** : étirer une vignette de $32 \times 32$ pixels vers $224 \times 224$ nécessite de multiplier artificiellement le nombre de pixels par $49$ par interpolation bilinéaire ou bicubique. Cela produit une image floue et pixelisée sans aucun détail fin sur la matière du déchet (verre, papier, plastique).

In [ ]:
# 1) Filtrage sur les images valides (non corrompues)
images_lisibles = df_exploration[~df_exploration['est_corrompue']].copy()

# 2) Conversion en entiers (pour éviter le format '512.0x384.0')
images_lisibles['largeur'] = images_lisibles['largeur'].astype(int)
images_lisibles['hauteur'] = images_lisibles['hauteur'].astype(int)
images_lisibles['resolution'] = images_lisibles['largeur'].astype(str) + 'x' + images_lisibles['hauteur'].astype(str)

# 3) Calcul des extrêmes et des fréquences
res_counts = images_lisibles['resolution'].value_counts()

print("=== STATISTIQUES DE RÉSOLUTION ===")
print(f"Largeur minimale : {images_lisibles['largeur'].min()} px  |  Largeur maximale : {images_lisibles['largeur'].max()} px")
print(f"Hauteur minimale : {images_lisibles['hauteur'].min()} px  |  Hauteur maximale : {images_lisibles['hauteur'].max()} px")
print(f"\nDistribution des résolutions ({len(res_counts)} formats distincts) :")
display(res_counts.to_frame(name='Nombre d\'images'))

# 4) Détection des images ne respectant pas le seuil minimal de 64x64
SEUIL_MIN = 64
masque_trop_petite = (images_lisibles['largeur'] < SEUIL_MIN) | (images_lisibles['hauteur'] < SEUIL_MIN)
df_trop_petites = images_lisibles[masque_trop_petite].copy()

print(f"\nTotal d'images trop petites (< {SEUIL_MIN}x{SEUIL_MIN}) : {len(df_trop_petites)}")

### Inventaire des images trop petites (< 64×64)

Examinons en détail les images rejetées et leur répartition par classe.

In [ ]:
# 1) Affichage du tableau des images rejetées
display(df_trop_petites[['nom', 'classe', 'largeur', 'hauteur', 'resolution', 'taille_octets']])

print("\nRépartition des images trop petites par classe :")
display(df_trop_petites['classe'].value_counts())

# 2) Comparaison visuelle : Image standard vs Vignette sous-dimensionnée
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Sélection dynamique de la résolution la plus fréquente (512x384)
resolution_majoritaire = res_counts.index[0]
ex_normale = images_lisibles[images_lisibles['resolution'] == resolution_majoritaire].iloc[0]

with Image.open(ex_normale['chemin']) as img:
    ax1.imshow(img.convert('RGB'))
ax1.set_title(f"Format standard : {ex_normale['nom']}\n({ex_normale['resolution']})", fontsize=11)
ax1.axis('off')

# Image trop petite (< 64x64)
ex_petite = df_trop_petites.iloc[0]
with Image.open(ex_petite['chemin']) as img:
    ax2.imshow(img.convert('RGB'))
ax2.set_title(f"Format rejeté : {ex_petite['nom']}\n({ex_petite['resolution']})", fontsize=11, color='red')
ax2.axis('off')

plt.suptitle("Comparaison d'échelle et de richesse d'information", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Bilan de la Partie 4
1. **Format dominant** : la quasi-totalité du jeu de données ($1\,013$ images sur $1\,026$, soit $98{,}7\,\%$) est homogène à la résolution de **$512 \times 384$ pixels**.
2. **Images sous le seuil** : exactement **13 images** sont inférieures à la contrainte de $64 \times 64$ pixels :
   * $32 \times 32$ pixels : 5 images
   * $48 \times 32$ pixels : 4 images
   * $40 \times 40$ pixels : 4 images
3. **Répartition des rejets** : ces 13 vignettes sont réparties sur 4 classes (`glass`: 4, `cardboard`: 3, `metal`: 3, `paper`: 3).

**Décision pour le dataset nettoyé** : ces 13 images manquent d'information discriminante et seront **exclues de `data/cleaned/`** pour éviter d'entraîner le modèle sur des textures dégradées.

## Partie 5 – Détecter les différents canaux

### Objectif
Déterminer la répartition des images selon leur nombre de canaux de couleur ($1$, $3$ ou $4$ canaux).

### Pourquoi l'uniformité des canaux est vitale en Deep Learning ?
* **Contrainte de dimension des tenseurs** : un réseau de neurones convolutif (CNN) traite des tenseurs d'entrée de forme rigide $(B, H, W, C)$ où $B$ est la taille du lot (*batch size*), $H$ la hauteur, $W$ la largeur et $C$ le nombre de canaux (généralement $C = 3$ pour le rouge, vert, bleu).
* **Le piège du mini-lot (*Batch Collation*)** : si un générateur de données tente d'empiler une image standard à 3 canaux $(224, 224, 3)$ avec une image avec transparence à 4 canaux $(224, 224, 4)$, l'opération d'empilement plante immédiatement :
  ```text
  RuntimeError: stack expects each tensor to be equal size, but got [3, 224, 224] at index 0 and [4, 224, 224] at index 1

In [ ]:
# 1) Recensement précis des canaux et des modes colorimétriques sur les images lisibles
df_canaux = images_lisibles.copy()

# Extraction des bandes de couleur réelles (ex: 'R,G,B' ou 'R,G,B,A')
bandes_reelles = []
for chemin in df_canaux['chemin']:
    with Image.open(chemin) as img:
        bandes_reelles.append(','.join(img.getbands()))

df_canaux['bandes'] = bandes_reelles

# 2) Décompte par nombre de canaux
repartition_canaux = df_canaux['nb_canaux'].value_counts().sort_index()

print("=== RÉPARTITION PAR NOMBRE DE CANAUX ===")
for nb, total in repartition_canaux.items():
    pct = (total / len(df_canaux)) * 100
    print(f"• {int(nb)} canal/canaux : {total} images ({pct:.1f} %)")

# 3) Tableau croisé : Mode PIL vs Nombre de canaux
print("\n=== CROISEMENT MODES PIL ET NOMBRE DE CANAUX ===")
display(pd.crosstab(df_canaux['mode'], df_canaux['nb_canaux'], margins=True, margins_name="Total"))

### Inventaire des images non conventionnelles (1 et 4 canaux)

Isolons la liste des images qui ne respectent pas le standard RGB à 3 canaux.

In [ ]:
# 1) Filtrage des images hors standard RGB (3 canaux)
images_non_rgb = df_canaux[df_canaux['nb_canaux'] != 3].copy()

print(f"Total d'images à canaux atypiques : {len(images_non_rgb)}")
display(images_non_rgb[['nom', 'classe', 'format', 'mode', 'nb_canaux', 'bandes']].head(10))

# 2) Illustration comparative : Image 3 canaux (RGB) vs Image 4 canaux (RGBA avec transparence)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Exemple standard RGB
ex_rgb = df_canaux[df_canaux['mode'] == 'RGB'].iloc[0]
with Image.open(ex_rgb['chemin']) as img:
    ax1.imshow(img)
ax1.set_title(f"Standard RGB (3 canaux) : {ex_rgb['nom']}\nMode: {ex_rgb['mode']} | Bandes: {ex_rgb['bandes']}", fontsize=10)
ax1.axis('off')

# Exemple RGBA (canal Alpha)
ex_rgba = images_non_rgb[images_non_rgb['mode'] == 'RGBA'].iloc[0]
with Image.open(ex_rgba['chemin']) as img:
    ax2.imshow(img)
ax2.set_title(f"Canal Alpha (4 canaux) : {ex_rgba['nom']}\nMode: {ex_rgba['mode']} | Bandes: {ex_rgba['bandes']}", fontsize=10, color='darkorange')
ax2.axis('off')

plt.suptitle("Comparaison des structures colorimétriques", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Bilan de la Partie 5
L'audit des canaux met en évidence trois catégories d'images dans notre jeu de données :
1. **$1\,006$ images standard à 3 canaux ($98{,}1\,\%$)** : mode `RGB`, directement exploitables.
2. **$18$ images à 4 canaux ($1{,}8\,\%$)** : mode `RGBA` (fichiers PNG/JPG dotés d'une couche alpha de transparence superflue pour la classification de déchets).
3. **$2$ images à 1 canal ($0{,}2\,\%$)** : mode `P` (palette indexée 8-bit, issues des fichiers GIF `image-violet-512x384.gif`).

**Décision pour le pipeline** :
Ces $20$ fichiers non-RGB ne doivent pas être supprimés (ils contiennent des déchets valides), mais ils nécessitent une transformation d'harmonisation. En **Partie 10**, nous appliquerons la conversion obligatoire `.convert('RGB')` sur l'ensemble du dataset pour garantir que $100\,\%$ des images possèdent exactement $3$ canaux.

## Partie 6 – Détecter les doublons

### Objectif
Écrire et appliquer une fonction capable d'identifier les images strictement identiques au sein du dataset.

### Méthode : le hachage cryptographique (MD5)
Comparer les noms de fichiers est insuffisant car, comme le précise le sujet, **deux fichiers ayant des noms différents peuvent contenir exactement la même image** (par exemple `metal125.jpg` et `metal125po.jpg`).

Pour détecter les doublons de manière infaillible :
1. Nous calculons pour chaque fichier son empreinte numérique unique (**hash MD5**).
2. Deux fichiers ayant rigoureusement les mêmes pixels produiront exactement la même signature MD5.
3. Nous regroupons les images partageant la même empreinte pour identifier les redondances.

In [ ]:
import hashlib

def calculer_hash_image(chemin_fichier):
    """
    Calcule l'empreinte MD5 du contenu d'un fichier image.
    Retourne la chaîne hexadécimale du hash (ou None si fichier corrompu/illisible).
    """
    try:
        with open(chemin_fichier, 'rb') as f:
            # Lecture binaire du fichier
            return hashlib.md5(f.read()).hexdigest()
    except Exception:
        return None


# 1) Calcul du hash pour l'ensemble des images lisibles du dataset
hashes_images = []

for chemin in images_lisibles['chemin']:
    h = calculer_hash_image(chemin)
    if h is not None:
        p = Path(chemin)
        hashes_images.append({
            'nom': p.name,
            'classe': p.parent.name,
            'chemin': str(chemin),
            'hash_md5': h
        })

df_hashes = pd.DataFrame(hashes_images)

# 2) Identification des doublons (tous les fichiers dont le hash apparaît plus d'une fois)
masque_doublons = df_hashes.duplicated(subset=['hash_md5'], keep=False)
df_doublons = df_hashes[masque_doublons].sort_values(by='hash_md5').copy()

nb_groupes = df_doublons['hash_md5'].nunique()
nb_fichiers_doublons = len(df_doublons)
nb_redondances = nb_fichiers_doublons - nb_groupes

print(f"=== BILAN DE LA DÉTECTION DE DOUBLONS ===")
print(f"• Nombre d'empreintes dupliquées (groupes) : {nb_groupes}")
print(f"• Nombre total de fichiers impliqués       : {nb_fichiers_doublons}")
print(f"• Nombre de fichiers en surplus (doublons) : {nb_redondances}")

### Inventaire détaillé des paires de doublons

Examinons les fichiers qui partagent la même empreinte numérique. Nous distinguons :
* **Doublons intra-classe** : deux fichiers identiques classés dans le même dossier (ex. copie renommée) ;
* **Doublons inter-classes** : deux fichiers identiques placés dans deux dossiers de classes différentes.

In [ ]:
# Construction d'un tableau récapitulatif par paire de doublons
recap_paires = []

for hash_val, groupe in df_doublons.groupby('hash_md5'):
    fichiers = groupe.to_dict('records')
    f1 = fichiers[0]
    f2 = fichiers[1]

    type_doublon = "Intra-classe" if f1['classe'] == f2['classe'] else "Inter-classes (conflit)"

    recap_paires.append({
        'Hash (début)': hash_val[:8] + '...',
        'Fichier 1': f1['nom'],
        'Classe 1': f1['classe'],
        'Fichier 2': f2['nom'],
        'Classe 2': f2['classe'],
        'Type de doublon': type_doublon,
        'chemin_1': f1['chemin'],
        'chemin_2': f2['chemin']
    })

df_paires = pd.DataFrame(recap_paires)
display(df_paires[['Hash (début)', 'Fichier 1', 'Classe 1', 'Fichier 2', 'Classe 2', 'Type de doublon']])

# Visualisation côte à côte de 2 exemples de doublons
fig, axes = plt.subplots(2, 2, figsize=(10, 6))

for i, idx in enumerate([0, 2]):  # Affichage de 2 paires
    paire = df_paires.iloc[idx]

    with Image.open(paire['chemin_1']) as img1:
        axes[i, 0].imshow(img1.convert('RGB'))
    axes[i, 0].set_title(f"Original : {paire['Fichier 1']} ({paire['Classe 1']})", fontsize=10)
    axes[i, 0].axis('off')

    with Image.open(paire['chemin_2']) as img2:
        axes[i, 1].imshow(img2.convert('RGB'))
    axes[i, 1].set_title(f"Doublon : {paire['Fichier 2']} ({paire['Classe 2']})\n[{paire['Type de doublon']}]", fontsize=10, color='darkred')
    axes[i, 1].axis('off')

plt.suptitle("Validation visuelle de l'identité des doublons", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Bilan de la Partie 6
La fonction de hachage identifie exactement **15 paires de doublons** (impliquant $30$ fichiers, soit $15$ images redondantes en surplus) :
1. **10 paires intra-classe** : des fichiers ayant subi un renommage accidentel (ex. `metal125.jpg` et `metal125po.jpg`, `plastic36.jpg` et `plastic36rt.jpg`).
2. **5 paires inter-classes** : des images identiques présentes dans deux catégories distinctes :
   * `glass115.jpg` (classé en `glass`) et `metal91.jpg` (classé en `metal`)
   * `glass176.jpg` (classé en `glass`) et `plastic152.jpg` (classé en `plastic`)
   * Les images artificielles noires, blanches et violettes présentes simultanément dans plusieurs sous-dossiers.

## Partie 7 : Détecter les images mal classées

### Objectif
Réaliser un contrôle visuel systématique pour identifier les images affectées à la mauvaise classe (mauvais sous-dossier dans `data/raw/`).

### Pourquoi le contrôle visuel humain est indispensable ?
* **La limite des métriques automatiques** : un fichier image peut avoir une taille correcte, une résolution standard ($512 \times 384$) et trois canaux RGB tout en représentant le mauvais objet. Aucun calcul d'écart-type ou de hash ne peut deviner qu'une bouteille en plastique a été glissée dans le dossier `cardboard/`.
* **L'impact du bruit d'étiquetage (*Label Noise*)** : si un réseau convolutif apprend qu'une canette métallique est du « plastique », il ajuste ses filtres sur des reflets métalliques pour prédire du plastique. Ce bruit dégrade directement la matrice de confusion du futur modèle.

### Méthodologie d'investigation
Le contrôle visuel s'appuie sur deux pistes :
1. **Les conflits inter-classes découverts en Partie 6** : deux images identiques trouvées dans deux dossiers distincts indiquent qu'au moins l'une des deux est dans la mauvaise classe.
2. **L'inspection ciblée des fichiers atypiques** : examen des images présentant des variations de nommage ou issues d'un échantillonnage visuel de vérification par classe.

In [ ]:
# Liste des images suspectes identifiées lors du contrôle visuel et des conflits de doublons
images_suspectes = [
    {
        'nom': 'cardboard86d.jpg',
        'dossier_actuel': 'cardboard',
        'classe_reelle_observee': 'plastic',
        'motif_visuel': 'Bouteille en plastique transparent avec bouchon vert',
        'chemin': DOSSIER_RAW / 'cardboard' / 'cardboard86d.jpg'
    },
    {
        'nom': 'glass12er.jpg',
        'dossier_actuel': 'glass',
        'classe_reelle_observee': 'paper',
        'motif_visuel': 'Prospectus publicitaire / papier journal d\'épicerie',
        'chemin': DOSSIER_RAW / 'glass' / 'glass12er.jpg'
    },
    {
        'nom': 'plasticx199fd.jpg',
        'dossier_actuel': 'plastic',
        'classe_reelle_observee': 'metal',
        'motif_visuel': 'Canette de boisson en aluminium écrasée (Natural Light)',
        'chemin': DOSSIER_RAW / 'plastic' / 'plasticx199fd.jpg'
    },
    {
        'nom': 'metal91.jpg',
        'dossier_actuel': 'metal',
        'classe_reelle_observee': 'glass',
        'motif_visuel': 'Bouteille en verre transparent vue du dessus (identique à glass115.jpg)',
        'chemin': DOSSIER_RAW / 'metal' / 'metal91.jpg'
    },
    {
        'nom': 'glass176.jpg',
        'dossier_actuel': 'glass',
        'classe_reelle_observee': 'plastic',
        'motif_visuel': 'Bouteille d\'eau en plastique rainurée (identique à plastic152.jpg)',
        'chemin': DOSSIER_RAW / 'glass' / 'glass176.jpg'
    }
]

df_mal_classees = pd.DataFrame(images_suspectes)
print(f"Nombre d'images mal classées identifiées : {len(df_mal_classees)}")
display(df_mal_classees[['nom', 'dossier_actuel', 'classe_reelle_observee', 'motif_visuel']])

### Preuve visuelle des erreurs d'étiquetage

Affichons les images identifiées pour confronter leur étiquette actuelle (dossier source) avec leur réalité visuelle.

In [ ]:
# Affichage visuel des 5 images mal classées
fig, axes = plt.subplots(1, len(images_suspectes), figsize=(16, 4))

for ax, img_info in zip(axes, images_suspectes):
    with Image.open(img_info['chemin']) as img:
        ax.imshow(img.convert('RGB'))

    titre = (
        f"{img_info['nom']}\n"
        f"Dossier : {img_info['dossier_actuel']} (FAUX)\n"
        f"Réel : {img_info['classe_reelle_observee']}"
    )
    ax.set_title(titre, fontsize=9, color='darkred' if img_info['dossier_actuel'] != img_info['classe_reelle_observee'] else 'green')
    ax.axis('off')

plt.suptitle("Contrôle visuel des erreurs d'affectation de classe", fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

### Bilan de la Partie 7
Le contrôle visuel permet d'isoler formellement **5 images mal classées** dans le dataset d'origine :
1. `cardboard86d.jpg` : classée à tort en `cardboard` alors qu'il s'agit d'une bouteille en **plastique**.
2. `glass12er.jpg` : classée à tort en `glass` alors qu'il s'agit d'un prospectus en **papier**.
3. `plasticx199fd.jpg` : classée à tort en `plastic` alors qu'il s'agit d'une canette en **métal**.
4. `metal91.jpg` : classée à tort en `metal` alors qu'il s'agit d'une bouteille en **verre** (doublon avec `glass115.jpg`).
5. `glass176.jpg` : classée à tort en `glass` alors qu'il s'agit d'une bouteille en **plastique** (doublon avec `plastic152.jpg`).

Cette vérification confirme l'existence d'erreurs d'attribution sémantique qu'aucun filtre statistique automatisé n'aurait pu corriger seul.

## Partie 8 : Analyser le déséquilibre des classes

### Objectif
Déterminer le nombre exact d'images par classe et mesurer le déséquilibre de distribution entre les différentes catégories de déchets.

### Pourquoi mesurer le déséquilibre des classes ?
* **Comportement des fonctions de perte (Loss)** : lorsqu'une classe est fortement sous-représentée, un modèle de classification optimise plus facilement son exactitude globale en apprenant à prédire presque systématiquement les classes majoritaires, ignorant la classe rare.
* **Évaluation de l'ampleur du déséquilibre** : calculer le ratio entre la classe la plus représentée et la classe la moins représentée permet d'objectiver le besoin de stratégies compensatoires.

In [ ]:
# 1) Décompte du nombre d'images par classe dans le dataset
comptage_classes = df_exploration['classe'].value_counts()
pourcentages = (df_exploration['classe'].value_counts(normalize=True) * 100).round(2)

df_repartition = pd.DataFrame({
    'Nombre d\'images': comptage_classes,
    'Pourcentage (%)': pourcentages
})

print("=== DISTRIBUTION DES CLASSES (DATASET BRUT) ===")
display(df_repartition)

classe_max = comptage_classes.idxmax()
classe_min = comptage_classes.idxmin()
ratio_desequilibre = comptage_classes.max() / comptage_classes.min()

print(f"\n• Total d'images répertoriées : {len(df_exploration)}")
print(f"• Classe majoritaire : '{classe_max}' ({comptage_classes.max()} images, {pourcentages[classe_max]} %)")
print(f"• Classe minoritaire : '{classe_min}' ({comptage_classes.min()} images, {pourcentages[classe_min]} %)")
print(f"• Ratio de déséquilibre (Max / Min) : {ratio_desequilibre:.2f}")

### Visualisation graphique de la distribution

Affichons l'histogramme des effectifs par classe pour observer visuellement les écarts de volume.

In [ ]:
# Visualisation graphique de la répartition des classes
plt.figure(figsize=(9, 4.5))

couleurs = ['#5bc0de' if c != 'trash' else '#d9534f' for c in comptage_classes.index]
barres = plt.bar(comptage_classes.index, comptage_classes.values, color=couleurs, edgecolor='black', linewidth=0.8)

plt.title("Répartition du nombre d'images par classe de déchet", fontsize=12, fontweight='bold', pad=12)
plt.xlabel("Classes", fontsize=11)
plt.ylabel("Nombre d'images", fontsize=11)
plt.grid(axis='y', linestyle=':', alpha=0.6)

# Ajout du nombre au-dessus de chaque barre
for barre in barres:
    hauteur = barre.get_height()
    plt.text(barre.get_x() + barre.get_width() / 2, hauteur + 4, str(hauteur), ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.ylim(0, comptage_classes.max() + 30)
plt.tight_layout()
plt.show()

### Génération du fichier de synthèse de l'audit : `reports/audit_images.csv`

Conformément à la consigne (NB 2 du sujet), nous synthétisons l'ensemble des résultats obtenus lors des Parties 1 à 8 dans le fichier récapitulatif `reports/audit_images.csv`.

In [ ]:
# Création du dossier reports s'il n'existe pas
DOSSIER_REPORTS = Path('../reports') if Path('../reports').exists() else Path('reports')
DOSSIER_REPORTS.mkdir(parents=True, exist_ok=True)
chemin_rapport = DOSSIER_REPORTS / 'audit_images.csv'

# Synthèse des métriques d'audit des Parties 1 à 8
donnees_audit = [
    {'Indicateur': "total d'images", 'Valeur': len(df_exploration)},
    {'Indicateur': "Images corrompues", 'Valeur': int(df_exploration['est_corrompue'].sum())},
    {'Indicateur': "Images quasi vides", 'Valeur': len(df_vides)},
    {'Indicateur': "Images trop petites", 'Valeur': len(df_trop_petites)},
    {'Indicateur': "Doublons", 'Valeur': nb_redondances},
    {'Indicateur': "Images grayscale", 'Valeur': int((df_canaux['nb_canaux'] == 1).sum())},
    {'Indicateur': "Images RGBA", 'Valeur': int((df_canaux['nb_canaux'] == 4).sum())},
    {'Indicateur': "images mal classées", 'Valeur': len(df_mal_classees)}
]

df_audit_synthese = pd.DataFrame(donnees_audit)
df_audit_synthese.to_csv(chemin_rapport, index=False, encoding='utf-8')

print(f"Rapport d'audit sauvegardé avec succès dans : {chemin_rapport.resolve()}\n")
display(df_audit_synthese)

### Bilan de la Partie 8
1. **Déséquilibre caractérisé** : la classe `paper` est la plus abondante ($252$ images, soit $24{,}4\,\%$) tandis que la classe `trash` est sévèrement minoritaire ($50$ images, soit $4{,}8\,\%$).
2. **Facteur d'échelle** : le ratio de déséquilibre est de **$5{,}04$**, ce qui signifie qu'il y a $5$ fois plus d'exemples de papier que de déchets tout-venant (`trash`).
3. **Audit du dataset brut finalisé** : le fichier `reports/audit_images.csv` enregistre l'ensemble des anomalies recensées (6 corrompues, 4 vides, 13 trop petites, 15 doublons, 2 monicanal, 18 RGBA et 5 mal classées).

## Partie 9 : Redimensionnement (224x224 avec conservation des proportions et padding)

### Objectif
Mettre à l'échelle les images à la résolution standard attendue par les modèles de vision par ordinateur ($224 \times 224$ pixels) tout en préservant scrupuleusement le ratio d'aspect initial grâce à une technique de remplissage neutre (*letterboxing / padding*).

### Pourquoi la conservation des proportions est-elle essentielle ?
* **Taille fixe requise par les CNN** : les architectures convolutives (ResNet, EfficientNet, MobileNet, etc.) imposent une taille d'entrée fixe (usuellement $224 \times 224$ ou $256 \times 256$) pour figer la dimension spatiale des tenseurs et des couches denses de classification.
* **Le danger du redimensionnement naïf (*squash / stretch*)** :
  * Si l'on force directement une image rectangulaire ($512 \times 384$, ratio $4:3$) dans un cadre carré ($224 \times 224$, ratio $1:1$) par simple étirement, l'objet subit une distorsion géométrique artificielle (anisotropie).
  * Une bouteille fine apparaît anormalement écrasée ou trapue, une boîte circulaire devient ovale. Le réseau de neurones apprend alors des représentations déformées qui ne correspondent pas à la réalité physique des objets.

### Solution technique : redimensionnement avec padding (*letterboxing*)
1. **Mise à l'échelle isotrope** : calcul du facteur d'échelle maximal permettant d'inscrire l'image dans $224 \times 224$ sans dépasser et sans altérer son ratio largeur/hauteur (avec le filtre d'interpolation de haute qualité `LANCZOS`).
2. **Centrage et padding** : l'image mise à l'échelle (par exemple $224 \times 168$) est positionnée au centre d'un canevas carré de $224 \times 224$ pixels. Les bandes vides périphériques sont complétées par une couleur neutre constante (pixels noirs `(0, 0, 0)`).

In [ ]:
from PIL import ImageOps

def redimensionner_avec_padding(image, taille_cible=(224, 224), couleur=(0, 0, 0)):
    """
    Redimensionne une image PIL à la taille cible (224x224) en conservant son ratio d'aspect
    et en ajoutant un padding neutre (bandes noires / letterboxing) pour combler l'espace restant.

    Paramètres :
        image : PIL.Image - L'image source à redimensionner.
        taille_cible : tuple (int, int) - Dimensions finales souhaitées (largeur, hauteur).
        couleur : tuple ou int - Couleur de remplissage pour le padding (noir par défaut).

    Retourne :
        PIL.Image : Nouvelle image aux dimensions exactes (taille_cible).
    """
    # Adaptation de la valeur de couleur si l'image est en niveaux de gris (mode 'L' ou '1')
    couleur_fond = 0 if image.mode in ('L', '1') else couleur

    # ImageOps.pad calcule l'échelle conservant le ratio, applique le filtre et centre sur le canevas
    return ImageOps.pad(image, taille_cible, method=Image.Resampling.LANCZOS, color=couleur_fond)

### Comparaison visuelle : Redimensionnement naïf (déformation) vs Redimensionnement avec padding

Confrontons les deux approches sur une image représentative du dataset ($512 \times 384$) pour observer directement l'impact sur la morphologie de l'objet.

In [ ]:
# Sélection d'une image représentative au format standard 512x384 (ex: bouteille de verre)
chemin_exemple = DOSSIER_RAW / 'glass' / 'glass1.jpg'

with Image.open(chemin_exemple) as img_source:
    img_originale = img_source.copy()

# 1. Redimensionnement naïf direct (étirement / écrasement géométrique)
img_naif = img_originale.resize((224, 224), resample=Image.Resampling.LANCZOS)

# 2. Redimensionnement avec préservation du ratio d'aspect et padding
img_padded = redimensionner_avec_padding(img_originale, taille_cible=(224, 224))

# Visualisation comparative
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# Image d'origine
axes[0].imshow(img_originale)
axes[0].set_title(f"1. Image d'origine\n{img_originale.size[0]}x{img_originale.size[1]} (Ratio 4:3 = 1.33)", fontsize=10, fontweight='bold')
axes[0].axis('off')

# Redimensionnement naïf
axes[1].imshow(img_naif)
axes[1].set_title(f"2. Redimensionnement naïf (Stretch)\n{img_naif.size[0]}x{img_naif.size[1]} (Distorsion visible)", fontsize=10, fontweight='bold', color='darkred')
axes[1].axis('off')

# Redimensionnement avec padding
axes[2].imshow(img_padded)
axes[2].set_title(f"3. Redimensionnement avec padding\n{img_padded.size[0]}x{img_padded.size[1]} (Proportions préservées)", fontsize=10, fontweight='bold', color='green')
axes[2].axis('off')

plt.suptitle("Impact du mode de redimensionnement sur la géométrie de l'objet", fontsize=13, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print(f"• Dimensions d'origine : {img_originale.size} | Ratio : {img_originale.size[0]/img_originale.size[1]:.2f}")
print(f"• Dimensions naïves    : {img_naif.size}       | Ratio forcé : {img_naif.size[0]/img_naif.size[1]:.2f} (objet déformé/écrasé)")
print(f"• Dimensions padded    : {img_padded.size}     | Proportions conservées avec bandes de padding neutre")

### Bilan de la Partie 9
1. **Élimination de la distorsion géométrique** : le redimensionnement naïf modifie la morphologie des objets (écrasement ou étirement), introduisant un biais géométrique artificiel.
2. **Standardisation $224 \times 224$** : la fonction `redimensionner_avec_padding()` garantit un format carré uniforme de $224 \times 224$ pixels tout en maintenant scrupuleusement l'intégrité des proportions d'origine.
3. **Robustesse multi-modes** : le padding s'adapte dynamiquement au mode de l'image (valeur scalaire pour les images en niveaux de gris, triplet RVB pour les images en couleur).

## Partie 10 : Uniformisation des canaux (Conversion en 3 canaux RGB)

### Objectif
Homogénéiser l'espace colorimétrique de toutes les images pour garantir que chaque échantillon du dataset possède strictement **3 canaux (Rouge, Vert, Bleu)**.

### Pourquoi l'uniformisation des canaux est-elle critique ?
* **Contrainte structurelle des réseaux de neurones (CNN)** :
  * Les tenseurs d'entrée attendus par les architectures de Computer Vision (ResNet, MobileNet, VGG, etc.) ont une dimension spatiale fixe avec 3 canaux : $(H, W, 3)$ ou $(B, H, W, 3)$ pour un batch.
  * Si un batchloader (`DataLoader` PyTorch ou `tf.data` TensorFlow) rencontre une image à 4 canaux (RGBA) ou à 1 canal (niveaux de gris 'L' ou palette 'P'), l'assemblage matriciel du mini-batch plante immédiatement avec une erreur de type `ValueError: cannot stack arrays of different shapes`.
* **Rappel de l'audit (Partie 5)** :
  * $1\,006$ images sont déjà en mode `RGB` (3 canaux).
  * $18$ images possèdent un canal alpha de transparence (`RGBA`, 4 canaux).
  * $2$ images sont encodées sous forme de palette indexée (`P` / GIF, 1 canal).

### Méthode de conversion
1. **Images RGBA (4 canaux)** : pour ne pas perdre l'information de transparence ou générer des artéfacts noirs, on compose l'image sur un fond blanc neutre en utilisant le canal Alpha comme masque de fusion, puis on extrait les 3 canaux RGB.
2. **Images Niveaux de gris ('L' ou '1')** : duplication du canal d'intensité unique sur les 3 composantes $(R=G=B)$, permettant de conserver l'apparence visuelle tout en respectant la dimensionnalité $(H, W, 3)$.
3. **Images Palette ('P' / GIF)** : déréférencement de la table de couleurs 8-bit vers l'espace 24-bit RGB.

In [ ]:
def uniformiser_en_rgb(image):
    """
    Convertit une image PIL en mode RGB standard à 3 canaux.
    
    Gère les différents espaces colorimétriques :
    - RGB : retournée directement sans altération.
    - RGBA : fusion de la transparence sur un fond blanc neutre via le canal Alpha.
    - L / 1 : duplication du canal d'intensité sur les 3 composantes (R=G=B).
    - P / GIF : conversion de la palette indexée vers l'espace RGB 24-bit.
    - CMYK : conversion de la quadrichromie vers le spectre RGB.
    
    Retourne :
        PIL.Image : Image au format strictement 'RGB' (3 canaux).
    """
    if image.mode == 'RGB':
        return image
    
    if image.mode == 'RGBA':
        # Fond blanc neutre pour gérer proprement la transparence
        fond_blanc = Image.new('RGB', image.size, (255, 255, 255))
        # Utilisation du 4ème canal (Alpha) comme masque de fusion
        fond_blanc.paste(image, mask=image.split()[3])
        return fond_blanc
    
    # Pour les modes 'L', '1', 'P' et 'CMYK'
    return image.convert('RGB')

### Validation et contrôle visuel avant / après conversion

Testons la fonction sur les deux cas atypiques identifiés lors de notre inventaire :
1. Une image avec canal Alpha : `cardboard/cardboard13v.png` (`RGBA`, 4 canaux).
2. Une image avec palette indexée : `cardboard/image-violet-512x384.gif` (`P`, 1 canal).

In [ ]:
# Sélection des fichiers de test atypiques identifiés dans data/raw
chemin_rgba = DOSSIER_RAW / 'cardboard' / 'cardboard13v.png'
chemin_p = DOSSIER_RAW / 'cardboard' / 'image-violet-512x384.gif'

cas_tests = [
    ('Image RGBA (4 canaux)', chemin_rgba),
    ('Image Palette P (1 canal)', chemin_p)
]

fig, axes = plt.subplots(len(cas_tests), 2, figsize=(10, 7))

for i, (label, chemin) in enumerate(cas_tests):
    with Image.open(chemin) as img_brute:
        img_originale = img_brute.copy()
        
    img_convertie = uniformiser_en_rgb(img_originale)
    
    arr_orig = np.array(img_originale)
    arr_conv = np.array(img_convertie)
    
    # Affichage avant
    axes[i, 0].imshow(img_originale)
    axes[i, 0].set_title(f"Avant : {chemin.name}\nMode: {img_originale.mode} | Shape: {arr_orig.shape}", fontsize=10, fontweight='bold', color='darkred')
    axes[i, 0].axis('off')
    
    # Affichage après
    axes[i, 1].imshow(img_convertie)
    axes[i, 1].set_title(f"Après : {chemin.name}\nMode: {img_convertie.mode} | Shape: {arr_conv.shape}", fontsize=10, fontweight='bold', color='green')
    axes[i, 1].axis('off')

plt.suptitle("Contrôle de l'uniformisation des canaux colorimétriques vers RGB", fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print("Vérification technique :")
for label, chemin in cas_tests:
    with Image.open(chemin) as img:
        conv = uniformiser_en_rgb(img)
        print(f"• {chemin.name:<28} : Mode initial = {img.mode:<4} -> Mode final = {conv.mode} | Canaux = {len(conv.getbands())}")

### Bilan de la Partie 10
1. **Homogénéité dimensionnelle absolue** : chaque image traitée par `uniformiser_en_rgb()` est garantie au format `RGB` (matrice $H \times W \times 3$).
2. **Préservation de la transparence** : la fusion sur fond blanc des images `RGBA` empêche l'apparition de contours sombres ou d'artéfacts de découpe.
3. **Compatibilité Deep Learning** : les tenseurs sont désormais parfaitement compatibles avec les pipelines d'ingestion (`tf.data` ou PyTorch `DataLoader`), éliminant tout risque de désaccord de dimensions lors de la création de mini-lots (*batches*).

## Partie 11 : Mise à l'échelle des pixels (Normalisation et Standardisation)

### Objectif
Transformer les valeurs brutes d'intensité lumineuse des pixels (initialement entières dans l'intervalle $[0, 255]$) en représentations numériques à virgule flottante (`float32`) adaptées à la descente de gradient et à l'optimisation des réseaux de neurones profonds.

### Pourquoi la mise à l'échelle des pixels est-elle indispensable en Deep Learning ?
1. **Éviter l'explosion / la saturation des gradients** :
   * Les matrices d'images brutes contiennent des valeurs entières entre $0$ et $255$ (`uint8`).
   * Multipliées par les poids initialisés aléatoirement des couches convolutives, de grandes valeurs d'entrée conduisent à des activations géantes qui saturent les fonctions non linéaires ou provoquent une instabilité numérique (*exploding / vanishing gradients*).
2. **Accélérer la convergence de la descente de gradient** :
   * Lorsque les variables d'entrée sont ramenées à une petite échelle (autour de $[0, 1]$ ou centrées réduites autour de $0$), les lignes de niveau de la surface de coût deviennent plus sphériques, permettant aux optimiseurs (Adam, SGD) de converger avec un pas d'apprentissage plus stable et rapide.

### Deux approches standards en vision par ordinateur
1. **Normalisation Min-Max ($[0, 1]$)** :
   $$X_{\text{norm}} = \frac{X}{255.0}$$
   Ramène simplement l'intensité de chaque pixel dans l'intervalle $[0.0, 1.0]$.
2. **Standardisation ImageNet (Z-score normalisation)** :
   $$X_{\text{std}} = \frac{X_{\text{norm}} - \mu}{\sigma}$$
   Utilise les moyennes $\mu = [0.485, 0.456, 0.406]$ et écarts-types $\sigma = [0.229, 0.224, 0.225]$ calculés sur les millions d'images du dataset ImageNet.
   > **Enjeu Transfer Learning** : si un modèle pré-entraîné (ResNet, MobileNet, EfficientNet) a été entraîné sur ImageNet, lui fournir des images centrées et réduites selon cette distribution exacte permet d'aligner immédiatement les filtres de convolution pré-entraînés avec nos données.

In [ ]:
def normaliser_pixels_minmax(image_array):
    """
    Normalise les pixels d'un tableau d'image dans l'intervalle [0.0, 1.0].
    
    Paramètre :
        image_array : np.ndarray ou PIL.Image
    Retourne :
        np.ndarray (float32) : Matrice avec valeurs dans [0.0, 1.0].
    """
    arr = np.array(image_array, dtype=np.float32)
    return arr / 255.0


def standardiser_pixels_imagenet(image_array):
    """
    Standardise les pixels selon la distribution ImageNet (centrage-réduction par canal).
    
    Paramètre :
        image_array : np.ndarray ou PIL.Image
    Retourne :
        np.ndarray (float32) : Matrice standardisée par canal.
    """
    # 1. Mise à l'échelle préalable dans [0.0, 1.0]
    arr_norm = normaliser_pixels_minmax(image_array)
    
    # 2. Vecteurs moyennes et écarts-types ImageNet (RGB)
    moyenne_imagenet = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std_imagenet = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    
    # 3. Centrage et réduction par canal (broadcasting NumPy)
    return (arr_norm - moyenne_imagenet) / std_imagenet

### Analyse comparative des distributions statistiques

Appliquons ces deux transformations à une image du dataset et observons le comportement statistique (min, max, moyenne, écart-type) ainsi que la forme des histogrammes d'intensité.

In [ ]:
# Sélection d'une image d'exemple
chemin_test = DOSSIER_RAW / 'glass' / 'glass1.jpg'
with Image.open(chemin_test) as img:
    img_rgb = uniformiser_en_rgb(img)
    arr_brut = np.array(img_rgb)

# Application des deux mises à l'échelle
arr_minmax = normaliser_pixels_minmax(arr_brut)
arr_imagenet = standardiser_pixels_imagenet(arr_brut)

# Tableau récapitulatif des statistiques
stats_df = pd.DataFrame([
    {
        'Transformation': 'Brut (uint8)',
        'Type': str(arr_brut.dtype),
        'Min': float(arr_brut.min()),
        'Max': float(arr_brut.max()),
        'Moyenne': round(float(arr_brut.mean()), 3),
        'Écart-type': round(float(arr_brut.std()), 3)
    },
    {
        'Transformation': 'Normalisation Min-Max [0, 1]',
        'Type': str(arr_minmax.dtype),
        'Min': round(float(arr_minmax.min()), 4),
        'Max': round(float(arr_minmax.max()), 4),
        'Moyenne': round(float(arr_minmax.mean()), 4),
        'Écart-type': round(float(arr_minmax.std()), 4)
    },
    {
        'Transformation': 'Standardisation ImageNet (Z-score)',
        'Type': str(arr_imagenet.dtype),
        'Min': round(float(arr_imagenet.min()), 4),
        'Max': round(float(arr_imagenet.max()), 4),
        'Moyenne': round(float(arr_imagenet.mean()), 4),
        'Écart-type': round(float(arr_imagenet.std()), 4)
    }
])

display(stats_df)

# Visualisation des histogrammes de distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
couleurs_canaux = ['red', 'green', 'blue']

# 1. Brut
for c, col in enumerate(couleurs_canaux):
    axes[0].hist(arr_brut[:, :, c].ravel(), bins=50, color=col, alpha=0.5, label=f'Canal {col.upper()[0]}')
axes[0].set_title("1. Pixels bruts [0, 255]\n(Distribution discrète uint8)", fontsize=10, fontweight='bold')
axes[0].set_xlabel("Valeur de pixel")
axes[0].set_ylabel("Fréquence")
axes[0].legend()
axes[0].grid(True, linestyle=':', alpha=0.5)

# 2. Min-Max
for c, col in enumerate(couleurs_canaux):
    axes[1].hist(arr_minmax[:, :, c].ravel(), bins=50, color=col, alpha=0.5, label=f'Canal {col.upper()[0]}')
axes[1].set_title("2. Normalisé [0, 1]\n(Mise à l'échelle linéaire float32)", fontsize=10, fontweight='bold', color='darkblue')
axes[1].set_xlabel("Valeur normalisée")
axes[1].legend()
axes[1].grid(True, linestyle=':', alpha=0.5)

# 3. ImageNet
for c, col in enumerate(couleurs_canaux):
    axes[2].hist(arr_imagenet[:, :, c].ravel(), bins=50, color=col, alpha=0.5, label=f'Canal {col.upper()[0]}')
axes[2].set_title("3. Standardisé ImageNet\n(Centré-réduit Z-score float32)", fontsize=10, fontweight='bold', color='darkgreen')
axes[2].set_xlabel("Valeur standardisée (Z-score)")
axes[2].legend()
axes[2].grid(True, linestyle=':', alpha=0.5)

plt.suptitle("Comparaison des distributions de pixels selon la méthode de mise à l'échelle", fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

### Bilan de la Partie 11
1. **Conversion en flottants `float32`** : qu'il s'agisse de Min-Max ou de standardisation ImageNet, le passage de `uint8` en `float32` est le préalable indispensable pour tout calcul matriciel différentiable.
2. **Normalisation $[0, 1]$** : adaptée aux architectures entraînées *from scratch* ou aux modèles légers où les couches de tête initialisent leurs filtres sur des valeurs positives bornées.
3. **Standardisation ImageNet** : standard de référence pour le *Transfer Learning* (modèles torchvision / keras pre-trained) afin de respecter fidèlement l'espace latent appris lors de l'entraînement sur les millions d'images d'ImageNet.

## Partie 12 : Découpage Train / Validation / Test

### Objectif
1. **Constitution du jeu de données nettoyé (`data/cleaned/`)** : matérialiser sur le disque les images filtrées de toutes les anomalies (corrompues, vides, sous-dimensionnées, doublons et erreurs d'étiquetage corrigées), redimensionnées à $224 \times 224$ avec padding et converties en 3 canaux RGB.
2. **Découpage stratifié (*Stratified Split*)** : partitionner l'ensemble nettoyé en 3 sous-ensembles indépendants :
   * **Train ($70\,\%$)** : entraînement des filtres et paramètres du réseau de neurones.
   * **Validation ($15\,\%$)** : ajustement des hyperparamètres, sélection du modèle et arrêt anticipé (*early stopping*).
   * **Test ($15\,\%$)** : évaluation finale objective et impartiale des performances sur des données jamais vues.

### Pourquoi la stratification est-elle indispensable ici ?
* **Impact du déséquilibre** : la classe minoritaire `trash` ne contient que $49$ images valides ($4{,}9\,\%$ du dataset nettoyé).
* **Risque d'un échantillonnage aléatoire simple** : un tirage purement aléatoire non stratifié risquerait de placer la quasi-totalité des images `trash` dans le train, laissant le jeu de test avec zéro ou trop peu d'échantillons pour mesurer une précision statistiquement fiable.
* **Garantie de la stratification** : elle reproduit strictement la proportion de chaque classe dans chacun des trois sous-ensembles.

In [ ]:
import shutil
import hashlib
from PIL import ImageOps

DOSSIER_CLEANED = Path('../data/cleaned') if Path('../data/cleaned').exists() else Path('data/cleaned')
DOSSIER_SPLITS = Path('../data/splits') if Path('../data/splits').exists() else Path('data/splits')

# 1. Préparation de l'arborescence data/cleaned/
classes = sorted([d.name for d in DOSSIER_RAW.iterdir() if d.is_dir()])
for c in classes:
    (DOSSIER_CLEANED / c).mkdir(parents=True, exist_ok=True)

# 2. Définition des règles d'exclusion et de correction établies lors de l'audit
corrompus_noms = {'cardboard83.jpg', 'glass74.jpg', 'metal48.jpg', 'paper213.jpg', 'plastic13.jpg', 'trash3.jpg'}
vides_noms = {'image-noire-512x384.png', 'image-blanche-512x384.jpg'}
corrections_classes = {
    'cardboard86d.jpg': 'plastic',
    'glass12er.jpg': 'paper',
    'plasticx199fd.jpg': 'metal',
    'metal91.jpg': 'glass',
    'glass176.jpg': 'plastic'
}

# 3. Filtrage, dédoublonnage et préparation des images
md5_vus = {}
images_a_traiter = []

fichiers_tries = sorted(list(DOSSIER_RAW.glob('*/*.*')), key=lambda x: str(x))

for chemin in fichiers_tries:
    nom = chemin.name
    # Exclusion des corrompues et vides
    if nom in corrompus_noms or nom in vides_noms:
        continue
    
    # Exclusion des images sous-dimensionnées (< 64x64)
    try:
        with Image.open(chemin) as img:
            if img.size[0] < 64 or img.size[1] < 64:
                continue
    except Exception:
        continue
    
    # Détection et exclusion des doublons MD5
    with open(chemin, 'rb') as f:
        h = hashlib.md5(f.read()).hexdigest()
    if h in md5_vus:
        continue
    md5_vus[h] = chemin
    
    # Correction de l'étiquetage si applicable
    classe_finale = corrections_classes.get(nom, chemin.parent.name)
    images_a_traiter.append((chemin, nom, classe_finale))

print(f"Nombre total d'images propres validées pour data/cleaned/ : {len(images_a_traiter)}")

# 4. Traitement et sauvegarde physique dans data/cleaned/ (Padding 224x224 + Conversion RGB)
for chemin_src, nom, classe_dest in images_a_traiter:
    with Image.open(chemin_src) as img_brute:
        img_rgb = uniformiser_en_rgb(img_brute)
        img_224 = redimensionner_avec_padding(img_rgb, taille_cible=(224, 224), couleur=(0, 0, 0))
        
        chemin_sauvegarde = DOSSIER_CLEANED / classe_dest / (Path(nom).stem + '.jpg')
        img_224.save(chemin_sauvegarde, format='JPEG', quality=95)

print("Sauvegarde terminée dans data/cleaned/.")

### Partitionnement stratifié (70% Train, 15% Validation, 15% Test)

Implémentons la fonction de découpage stratifié afin d'obtenir une répartition rigoureusement proportionnelle pour chaque classe de déchet.

In [ ]:
# 1. Recensement des images nettoyées
images_cleaned = []
for p in DOSSIER_CLEANED.glob('*/*.jpg'):
    images_cleaned.append({
        'chemin': p,
        'nom': p.name,
        'classe': p.parent.name
    })

df_cleaned = pd.DataFrame(images_cleaned)

# 2. Algorithme de partitionnement stratifié (reproductible avec seed=42)
def partitionner_stratifie(df, colonne_classe='classe', ratio_train=0.70, ratio_val=0.15, seed=42):
    train_list, val_list, test_list = [], [], []
    
    for classe, groupe in df.groupby(colonne_classe):
        groupe_melange = groupe.sample(frac=1, random_state=seed)
        n = len(groupe_melange)
        n_train = int(round(n * ratio_train))
        n_val = int(round(n * ratio_val))
        
        g_train = groupe_melange.iloc[:n_train].copy()
        g_train['split'] = 'train'
        train_list.append(g_train)
        
        g_val = groupe_melange.iloc[n_train:n_train + n_val].copy()
        g_val['split'] = 'val'
        val_list.append(g_val)
        
        g_test = groupe_melange.iloc[n_train + n_val:].copy()
        g_test['split'] = 'test'
        test_list.append(g_test)
        
    return pd.concat(train_list), pd.concat(val_list), pd.concat(test_list)

df_train, df_val, df_test = partitionner_stratifie(df_cleaned, seed=42)
df_splits = pd.concat([df_train, df_val, df_test]).reset_index(drop=True)

# 3. Création des dossiers physiques dans data/splits/
for split in ['train', 'val', 'test']:
    for c in classes:
        (DOSSIER_SPLITS / split / c).mkdir(parents=True, exist_ok=True)

for _, row in df_splits.iterrows():
    dest = DOSSIER_SPLITS / row['split'] / row['classe'] / row['nom']
    shutil.copy2(row['chemin'], dest)

# 4. Sauvegarde du fichier de métadonnées du split
chemin_rapport_splits = DOSSIER_REPORTS / 'repartition_splits.csv'
df_splits[['nom', 'classe', 'split']].to_csv(chemin_rapport_splits, index=False, encoding='utf-8')

# 5. Synthèse des volumes par sous-ensemble
recap_splits = pd.crosstab(df_splits['classe'], df_splits['split'], margins=True)
recap_splits = recap_splits[['train', 'val', 'test', 'All']]
print(f"Découpage terminé avec succès ! Rapport sauvegardé dans : {chemin_rapport_splits}")
display(recap_splits)

### Visualisation de la distribution stratifiée

Affichons la répartition des effectifs par classe à travers les trois sous-ensembles (Train, Validation, Test) pour attester visuellement du respect des proportions.

In [ ]:
# Tableau de comptage pour le graphique
recap_plot = pd.crosstab(df_splits['classe'], df_splits['split'])[['train', 'val', 'test']]

ax = recap_plot.plot(kind='bar', figsize=(11, 5), colormap='viridis', edgecolor='black', width=0.75)
plt.title("Répartition stratifiée des classes d'images (Train / Validation / Test)", fontsize=13, fontweight='bold', pad=15)
plt.xlabel("Classes de déchets", fontsize=11)
plt.ylabel("Nombre d'images", fontsize=11)
plt.grid(axis='y', linestyle=':', alpha=0.6)
plt.xticks(rotation=0)
plt.legend(title="Sous-ensemble", fontsize=10)

# Ajout des valeurs au-dessus des barres
for p in ax.patches:
    h = p.get_height()
    if h > 0:
        ax.annotate(str(int(h)), (p.get_x() + p.get_width() / 2., h + 2), ha='center', va='bottom', fontsize=8)

plt.ylim(0, recap_plot['train'].max() + 25)
plt.tight_layout()
plt.show()

print(f"• Total Train : {len(df_train)} images ({len(df_train)/len(df_splits)*100:.1f} %)")
print(f"• Total Val   : {len(df_val)} images ({len(df_val)/len(df_splits)*100:.1f} %)")
print(f"• Total Test  : {len(df_test)} images ({len(df_test)/len(df_splits)*100:.1f} %)")
print(f"• Total global: {len(df_splits)} images")

### Bilan de la Partie 12
1. **Dataset nettoyé intègre (`data/cleaned/`)** : $996$ images de haute qualité, exemptes de fichiers corrompus, de vignettes vides, de résolutions inadéquates ou de doublons, standardisées en $224 \times 224$ pixels et $3$ canaux RGB.
2. **Partitionnement équilibré** : $696$ images en Train ($69{,}9\,\%$), $149$ en Validation ($15{,}0\,\%$) et $151$ en Test ($15{,}1\,\%$).
3. **Représentativité de la classe minoritaire** : la classe `trash` compte exactement $34$ images en Train, $7$ en Val et $8$ en Test, garantissant qu'elle pourra être entraînée et évaluée sans biais d'absence.
4. **Traçabilité complète** : le mapping individuel de chaque image est consigné dans `reports/repartition_splits.csv`.

## Partie 13 : Data Augmentation (Augmentation de données)

### Objectifs
1. **Compenser le déséquilibre de la classe minoritaire (`trash`)** dans le jeu d'entraînement (`train`) en générant des variations réalistes d'images.
2. **Implémenter les transformations de vision par ordinateur recommandées par le sujet** : rotation légère, retournement horizontal, zoom, translation, variations de luminosité, de contraste et de couleur.
3. **Répondre à la question méthodologique fondamentale** : *Pourquoi éviter formellement de faire la « data augmentation » avant le découpage du dataset ?*

---

### Question 2 du sujet : Pourquoi éviter de faire la « data augmentation » avant le découpage du dataset ?

Effectuer la *data augmentation* avant le découpage du dataset constitue une erreur méthodologique grave en Machine Learning / Deep Learning appelée **fuite de données (*Data Leakage*)**. Voici pourquoi :

1. **La fuite d'information (*Data Leakage*)** :
   * Si une image originale subit une légère rotation ou un changement de luminosité *avant* le partitionnement, l'image originale et ses variantes dérivées risquent de se retrouver réparties de part et d'autre : l'une dans le jeu d'entraînement (*Train*), l'autre dans le jeu de test (*Test*).
   * Le réseau de neurones n'aura alors aucune difficulté à classifier correctement l'image du test, non pas parce qu'il sait reconnaître le déchet, mais parce qu'il a déjà mémorisé ses pixels ou son arrière-plan quasi identique lors de l'entraînement.
2. **Surévaluation artificielle des performances (*Over-optimistic Evaluation*)** :
   * Les métriques de précision (Accuracy, F1-Score) mesurées sur le jeu de test seront artificiellement et trompeusement élevées.
   * En condition réelle de production, confronté à une image d'un déchet véritablement nouveau, le modèle échouera, révélant un surapprentissage (*overfitting*) sévère qui avait été masqué.
3. **Règle d'or en Science des Données** :
   * Le jeu de test (et de validation) doit **toujours** rester représentatif de la réalité brute du terrain (*unadulterated ground truth*).
   * Par conséquent, la **Data Augmentation ne doit s'appliquer EXCLUSIVEMENT qu'au sous-ensemble d'entraînement (`Train`)**.

In [ ]:
import random
from PIL import Image, ImageEnhance, ImageOps

# 1. Définition des transformations Keras et équivalent natif
# En Keras / TensorFlow, la pipeline séquentielle s'écrit classiquement :
"""
import keras
from keras import layers

pipeline_augmentation_keras = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(factor=0.05),                                # Rotation +/- 18 degrés
    layers.RandomZoom(height_factor=(-0.1, 0.1)),                       # Zoom in/out de 10%
    layers.RandomTranslation(height_factor=0.05, width_factor=0.05),   # Translation de 5%
    layers.RandomBrightness(factor=0.15),                               # Luminosité +/- 15%
    layers.RandomContrast(factor=0.15),                                 # Contraste +/- 15%
], name="augmentation_classe_minoritaire")
"""

# Implémentation universelle des transformations pour une compatibilité totale sans dépendance lourde
def appliquer_transformations_augmentation(image):
    """
    Applique un ensemble de transformations aléatoires réalistes à une image PIL 224x224 :
    - Retournement horizontal (flip)
    - Rotation légère (-15° à +15°)
    - Zoom / crop ou padding (0.9 à 1.1)
    - Translation légère
    - Variation de luminosité (0.85 à 1.15)
    - Variation de contraste (0.85 à 1.15)
    - Variation de saturation des couleurs (0.85 à 1.15)
    """
    img_aug = image.copy()
    w, h = img_aug.size
    
    # 1. Retournement horizontal aléatoire (probabilité 50%)
    if random.random() > 0.5:
        img_aug = img_aug.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        
    # 2. Rotation légère (-15° à +15°)
    angle = random.uniform(-15, 15)
    img_aug = img_aug.rotate(angle, resample=Image.Resampling.BILINEAR, fillcolor=(0, 0, 0))
    
    # 3. Zoom / Translation légère
    facteur_zoom = random.uniform(0.92, 1.08)
    new_w, new_h = int(w * facteur_zoom), int(h * facteur_zoom)
    img_redim = img_aug.resize((new_w, new_h), resample=Image.Resampling.BILINEAR)
    
    if facteur_zoom > 1.0:
        # Recadrage central
        gauche = (new_w - w) // 2
        haut = (new_h - h) // 2
        img_aug = img_redim.crop((gauche, haut, gauche + w, haut + h))
    else:
        # Complément avec padding noir
        img_aug = ImageOps.pad(img_redim, (w, h), color=(0, 0, 0))
        
    # 4. Variation de luminosité (+/- 15%)
    facteur_lum = random.uniform(0.85, 1.15)
    img_aug = ImageEnhance.Brightness(img_aug).enhance(facteur_lum)
    
    # 5. Variation de contraste (+/- 15%)
    facteur_contraste = random.uniform(0.85, 1.15)
    img_aug = ImageEnhance.Contrast(img_aug).enhance(facteur_contraste)
    
    # 6. Légère variation de saturation des couleurs
    facteur_couleur = random.uniform(0.85, 1.15)
    img_aug = ImageEnhance.Color(img_aug).enhance(facteur_couleur)
    
    return img_aug

### Visualisation des transformations sur un échantillon de la classe `trash`

Observons les différentes variantes synthétisées à partir d'une même image d'origine de la classe minoritaire pour vérifier la pertinence et le réalisme des modifications.

In [ ]:
# Sélection d'une image de déchet de la classe trash dans le Train set
dossier_trash_train = DOSSIER_SPLITS / 'train' / 'trash'
images_trash = sorted(list(dossier_trash_train.glob('*.jpg')))
image_reference = Image.open(images_trash[0])

# Génération de 5 variantes augmentées
random.seed(42)
variantes = [appliquer_transformations_augmentation(image_reference) for _ in range(5)]

# Affichage comparatif
fig, axes = plt.subplots(1, 6, figsize=(18, 4))

# Image originale
axes[0].imshow(image_reference)
axes[0].set_title("Originale (Train)", fontsize=10, fontweight='bold', color='blue')
axes[0].axis('off')

# Variantes augmentées
for idx, (ax, img_v) in enumerate(zip(axes[1:], variantes), start=1):
    ax.imshow(img_v)
    ax.set_title(f"Variante #{idx}\n(Rot/Flip/Zoom/Lum)", fontsize=9, fontweight='bold', color='darkgreen')
    ax.axis('off')

plt.suptitle("Démonstration des transformations de Data Augmentation sur la classe 'trash'", fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Application de la Data Augmentation pour rééquilibrer la classe `trash` dans le Train set

Générons 3 variantes augmentées pour chaque image existante de la classe `trash` dans le jeu d'entraînement afin de porter son effectif de $34$ à $136$ images, réduisant ainsi drastiquement le ratio de déséquilibre initial.

In [ ]:
# 1. État initial de la classe trash dans train
nb_initial = len(images_trash)
nb_variantes_par_image = 3
nb_creees = 0

random.seed(42)

for chemin_img in images_trash:
    # Ne traiter que les images originales (pas celles déjà suffixées _aug)
    if '_aug' in chemin_img.stem:
        continue
        
    with Image.open(chemin_img) as img_base:
        for v in range(1, nb_variantes_par_image + 1):
            img_aug = appliquer_transformations_augmentation(img_base)
            nom_aug = f"{chemin_img.stem}_aug{v}.jpg"
            chemin_dest = dossier_trash_train / nom_aug
            img_aug.save(chemin_dest, format='JPEG', quality=95)
            nb_creees += 1

nb_final = len(list(dossier_trash_train.glob('*.jpg')))
print(f"=== BILAN DATA AUGMENTATION (TRAIN SET - CLASSE TRASH) ===")
print(f"• Effectif initial dans train/trash/ : {nb_initial} images")
print(f"• Images synthétisées par augmentation: {nb_creees} images")
print(f"• Nouvel effectif total dans train/trash/ : {nb_final} images")

# Actualisation des effectifs par classe dans le Train set
comptage_train_ajour = {c: len(list((DOSSIER_SPLITS / 'train' / c).glob('*.jpg'))) for c in classes}
df_train_ajour = pd.DataFrame(list(comptage_train_ajour.items()), columns=['Classe', 'Nb Images (Train)'])
df_train_ajour = df_train_ajour.sort_values(by='Nb Images (Train)', ascending=False).reset_index(drop=True)

display(df_train_ajour)

### Bilan de la Partie 13
1. **Équilibre restauré** : grâce à l'injection de $102$ images augmentées, la classe `trash` passe de $34$ à **$136$ images** dans le sous-ensemble d'entraînement, s'alignant sur les volumes des autres classes ($98$ à $172$).
2. **Diversité et robustesse des filtres** : les variations géométriques et photométriques appliquées forcent le futur modèle à apprendre des invariants structurels (invariance aux rotations, réflexions et changements d'éclairage).
3. **Étanchéité méthodologique préservée** : l'augmentation a été strictement cantonnée au dossier `train/`. Les dossiers `val/` ($7$ images) et `test/` ($8$ images) n'ont subi aucune modification, garantissant une évaluation de performance $100\,\%$ intègre et sans fuite de données (*Zero Data Leakage*).